# Axial v3 Iteration A - raw_0 audit

Auditoria estructural train/validation y auditoria probabilistica validation-only. No interpreta `raw_0` como anatomia.

In [1]:
import importlib.util
import subprocess
import sys

required = {"numpy": "numpy", "pandas": "pandas", "PIL": "Pillow", "matplotlib": "matplotlib", "torch": "torch"}
missing = [pip_package for module_name, pip_package in required.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

In [2]:
import os
from pathlib import Path
import importlib

# Habilitar ejecución real
os.environ["PFI_RUN_AXIAL_V3_AUDIT"] = "1"

# Dataset y manifest curado
os.environ["PFI_DATASET_ROOT"] = "/content/drive/MyDrive/PFI_MVP"
os.environ["AXIAL_E9_CURATED_SPLIT_CSV"] = (
    "/content/drive/MyDrive/PFI_MVP/"
    "RUTA_REAL_DEL_MANIFEST_CURADO.csv"
)

# Checkpoint BEST de axial-final-v2
os.environ["AXIAL_V2_CHECKPOINT_PATH"] = (
    "/content/drive/MyDrive/PFI_MVP/outputs/"
    "axial_final_v2_training/resume/axial-final-v2/"
    "axial_t2_alkafri_v2.best_checkpoint.pt"
)

# Guardar resultados en Drive
os.environ["PFI_OUTPUT_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/outputs"
)

# Identificación de esta auditoría
os.environ["PFI_RUN_ID"] = "axial-v3-iteration-a-v2-audit"

# Etiquetas originales del dataset
os.environ["PFI_MASK_LABEL_MODE"] = "raw"

# Validación reproducible
os.environ["AXIAL_EXPECTED_V2_RUN_ID"] = "axial-final-v2"
os.environ["AXIAL_EXPECTED_AI_SERVICE_COMMIT"] = "285159"
os.environ["PFI_ALLOW_INCOMPLETE_V2_CHECKPOINT_METADATA"] = "0"

manifest_path = Path(os.environ["AXIAL_E9_CURATED_SPLIT_CSV"])
checkpoint_path = Path(os.environ["AXIAL_V2_CHECKPOINT_PATH"])

print({
    "manifestExists": manifest_path.is_file(),
    "manifestPath": str(manifest_path),
    "checkpointExists": checkpoint_path.is_file(),
    "checkpointPath": str(checkpoint_path),
})

{'manifestExists': False, 'manifestPath': '/content/drive/MyDrive/PFI_MVP/RUTA_REAL_DEL_MANIFEST_CURADO.csv', 'checkpointExists': True, 'checkpointPath': '/content/drive/MyDrive/PFI_MVP/outputs/axial_final_v2_training/resume/axial-final-v2/axial_t2_alkafri_v2.best_checkpoint.pt'}


In [3]:
import os

os.environ["PFI_REPO_REF"] = "2b8b943df73bd4a2f45ba4865158301d2d3889cf"
os.environ["PFI_REPO_ROOT"] = "/content/PFI_MVPTest_Enzo_AImodule"
os.environ["PFI_REPO_URL"] = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)

print({
    "PFI_REPO_REF": os.environ["PFI_REPO_REF"],
    "PFI_REPO_ROOT": os.environ["PFI_REPO_ROOT"],
})

{'PFI_REPO_REF': '2b8b943df73bd4a2f45ba4865158301d2d3889cf', 'PFI_REPO_ROOT': '/content/PFI_MVPTest_Enzo_AImodule'}


In [4]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule" if IN_COLAB else ".")).resolve()
REPO_REF = os.getenv("PFI_REPO_REF", "").strip()
REPO_URL = os.getenv("PFI_REPO_URL", "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git")
if not (REPO_ROOT / ".git").exists():
    if not REPO_REF:
        raise RuntimeError("PFI_REPO_REF debe definirse para clonar en un runtime nuevo")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_ROOT)])
if REPO_REF:
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF])
effective_sha = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
os.environ["PFI_REPO_ROOT"] = str(REPO_ROOT)
print({"repoRoot": str(REPO_ROOT), "effectiveSha": effective_sha})
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NameError: name 'IN_COLAB' is not defined

In [ ]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path(os.environ["PFI_REPO_ROOT"]).resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
import lumbar_mri.axial_v3.iteration_a as iteration_a

iteration_a = importlib.reload(iteration_a)

CFG = iteration_a.AxialV3AuditConfig()

print({
    "runId": CFG.RUN_ID,
    "manifest": str(CFG.SPLIT_MANIFEST_PATH),
    "checkpoint": str(CFG.V2_CHECKPOINT_PATH),
    "outputRoot": str(CFG.OUTPUT_ROOT),
})

In [ ]:
import json

result = iteration_a.run_iteration_a(CFG)

print(json.dumps(result, indent=2, default=str))

## Protocol

The runner validates storage, reads the curated manifest, filters development records to train/validation, maps raw labels explicitly to class indices, writes structural audit tables, and optionally evaluates the v2 checkpoint on validation only.

In [ ]:
RUN_AUDIT = os.getenv("PFI_RUN_AXIAL_V3_AUDIT", "0") == "1"

if RUN_AUDIT:
    result = run_iteration_a(CFG)
    print(json.dumps(result, indent=2, default=str))
else:
    print("Iteration A preparada. Definir PFI_RUN_AXIAL_V3_AUDIT=1 para ejecutar.")